In [44]:
import warnings
warnings.filterwarnings('ignore')
import pandas as pd
df = pd.read_excel("data/hotel_bookings.xlsx", sheet_name= "Sheet 1")

In [56]:
# Keep relevant columns for the analysis
cols_to_keep = [
    "hotel",
    "arrival_date_year",
    "arrival_date_month",
    "is_canceled",
    "lead_time",
    "stays_in_weekend_nights",
    "stays_in_week_nights",
    "adults",
    "children",
    "adr",
    "market_segment",
    "country"
]

df = df[cols_to_keep].copy()
print(df.shape)
print(df.dtypes)

(119390, 12)
hotel                       object
arrival_date_year            int64
arrival_date_month          object
is_canceled                  int64
lead_time                    int64
stays_in_weekend_nights      int64
stays_in_week_nights         int64
adults                       int64
children                   float64
adr                        float64
market_segment              object
country                     object
dtype: object


In [72]:
# Data quality check

print(df.isnull().sum())

print(df["hotel"].value_counts())

print(df["arrival_date_year"].value_counts())

print(df["adr"].describe())  

hotel                        0
arrival_date_year            0
arrival_date_month           0
is_canceled                  0
lead_time                    0
stays_in_weekend_nights      0
stays_in_week_nights         0
adults                       0
children                     4
adr                          0
market_segment               0
country                    488
length_of_stay               0
month_num                    0
dtype: int64
hotel
City Hotel      79330
Resort Hotel    40060
Name: count, dtype: int64
arrival_date_year
2016    56707
2017    40687
2015    21996
Name: count, dtype: int64
count    119390.000000
mean        101.831122
std          50.535790
min          -6.380000
25%          69.290000
50%          94.575000
75%         126.000000
max        5400.000000
Name: adr, dtype: float64


In [68]:
# Add "length_of_stay" column

df["length_of_stay"] = df["stays_in_weekend_nights"] + df["stays_in_week_nights"]

# Month order for sorting

month_order = {
    "January": 1, "February": 2, "March": 3, "April": 4,
    "May": 5, "June": 6, "July": 7, "August": 8,
    "September": 9, "October": 10, "November": 11, "December": 12
}
df["month_num"] = df["arrival_date_month"].map(month_order)

In [69]:
# Aggregate monthlly metrics based on hotel type

monthly = df.groupby(["hotel", "arrival_date_year", "arrival_date_month", "month_num"]).agg(
    total_bookings=("is_canceled", "count"),
    cancellations=("is_canceled", "sum"),
    avg_lead_time=("lead_time", "mean"),
    avg_adr=("adr", "mean"),
    avg_length_of_stay=("length_of_stay", "mean")
).reset_index()

monthly["cancellation_rate"] = monthly["cancellations"] / monthly["total_bookings"]
monthly = monthly.sort_values(["hotel", "arrival_date_year", "month_num"])

print(monthly.head(10))

         hotel  arrival_date_year arrival_date_month  month_num  \
2   City Hotel               2015               July          7   
0   City Hotel               2015             August          8   
5   City Hotel               2015          September          9   
4   City Hotel               2015            October         10   
3   City Hotel               2015           November         11   
1   City Hotel               2015           December         12   
10  City Hotel               2016            January          1   
9   City Hotel               2016           February          2   
13  City Hotel               2016              March          3   
6   City Hotel               2016              April          4   

    total_bookings  cancellations  avg_lead_time     avg_adr  \
2             1398            939     180.595136   69.819170   
0             2480           1232     113.636694   77.729198   
5             3529           1543     113.737603  101.064472   
4     

In [70]:
# Aggregate by market segment

segment = df.groupby(["hotel", "market_segment"]).agg(
    total_bookings=("is_canceled", "count"),
    cancellations=("is_canceled", "sum"),
    avg_adr=("adr", "mean"),
    avg_lead_time=("lead_time", "mean")
).reset_index()

segment["cancellation_rate"] = segment["cancellations"] / segment["total_bookings"]
segment = segment.sort_values("cancellation_rate", ascending=False)

print(segment.head(10))

           hotel market_segment  total_bookings  cancellations     avg_adr  \
7     City Hotel      Undefined               2              2   15.000000   
4     City Hotel         Groups           13975           9623   84.921885   
5     City Hotel  Offline TA/TO           16747           7173   93.017660   
11  Resort Hotel         Groups            5836           2474   66.446964   
6     City Hotel      Online TA           38748          14491  118.919533   
13  Resort Hotel      Online TA           17729           6248  113.432480   
0     City Hotel       Aviation             237             52  100.142110   
2     City Hotel      Corporate            2986            641   83.119980   
3     City Hotel         Direct            6093           1056  119.479682   
8   Resort Hotel  Complementary             201             33    3.657413   

    avg_lead_time  cancellation_rate  
7        1.500000           1.000000  
4      195.224830           0.688587  
5      141.497104       

In [71]:
monthly.to_excel("hotel_month_metrics.xlsx", index=False)
segment.to_excel("hotel_segment_metrics.xlsx", index=False)